# Análisis de Resultados

Objetivo: evaluar en profundidad los modelos seleccionados en la etapa de entrenamiento — uno por experimento — usando el dataset de trabajo.

Se analizan las matrices de confusión, las curvas ROC y Precision-Recall, la importancia de variables y el comportamiento del modelo ante distintos umbrales de clasificación.

## 1. Importar paquetes

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score,
    classification_report, f1_score
)
warnings.filterwarnings('ignore')

## 2. Rutas y carga de artefactos

Se resuelve la raíz del repositorio y se cargan los datasets preprocesados junto con los modelos entrenados en la etapa anterior.

In [ ]:
from pathlib import Path

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'README.md').exists():
    repo_root = repo_root.parent

if not (repo_root / 'README.md').exists():
    raise FileNotFoundError('No se encontró la raíz del repositorio (README.md).')

datos_dir    = repo_root / '02_Datos' / '01_Trabajo'
modelos_dir  = repo_root / '04_Modelos'
resultados_dir = repo_root / '05_Resultados'
resultados_dir.mkdir(parents=True, exist_ok=True)

print(f'Raíz del repositorio : {repo_root}')

In [ ]:
import pickle

with open(datos_dir / 'trabajo_preprocesado_moid.pickle', 'rb') as f:
    df_moid = pickle.load(f)

with open(datos_dir / 'trabajo_preprocesado.pickle', 'rb') as f:
    df_nomoid = pickle.load(f)

print(f'df_moid  : {df_moid.shape}')
print(f'df_nomoid: {df_nomoid.shape}')

In [ ]:
# Separación features / target
X_moid,   y_moid   = df_moid.drop(columns=['pha']),   df_moid['pha']
X_nomoid, y_nomoid = df_nomoid.drop(columns=['pha']), df_nomoid['pha']

# Cargar modelos — los nombres se resuelven dinámicamente desde 04_Modelos/
modelos_a = sorted(modelos_dir.glob('*_pha_A_v1_pipeline.joblib'))
modelos_b = sorted(modelos_dir.glob('*_pha_B_v1_pipeline.joblib'))

if not modelos_a or not modelos_b:
    raise FileNotFoundError('No se encontraron modelos en 04_Modelos/. Ejecutar 05_Entrenamiento primero.')

modelo_a = joblib.load(modelos_a[0])
modelo_b = joblib.load(modelos_b[0])

print(f'Modelo A: {modelos_a[0].name}')
print(f'Modelo B: {modelos_b[0].name}')

## 3. Validación cruzada estratificada

Se define el mismo `cv` utilizado en entrenamiento para garantizar que las métricas sean comparables.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)